# IC 4040 - *SBC* Image Reducer

<div class="alert alert-block alert-info">
    <b>Note:</b> This notebook should be run with the <span style="font-family: 'Ariel', monospace;">stenv</span> environment.
</div>

The purpose of this notebook is to reduce the FLC files from Hubble by:

1. Aligning FLCs to the GAIA catalog
2. Drizzling Images together from a particular filter

## Imports

In [ ]:
# Python Imports
import logging
import os
import re
import warnings
from pathlib import Path

# Astropy Colab Imports
from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.table import QTable, vstack
from drizzlepac import updatehdr
from drizzlepac.astrodrizzle import AstroDrizzle

# 3rd Party Imports
import numpy as np
from tqdm.notebook import tqdm


## Notebook Setup

In [ ]:
# Check Directory
if Path.cwd().name != "Images":
    if Path.cwd().name == "Notebooks":
        os.chdir("../../Images")
    else:
        raise RuntimeError("This notebook must be run from the Images directory.")

# Configure Logging
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)
logger.info("Current Directory: %s", Path.cwd())

In [ ]:
# Data Directory
DATA_DIR = Path('RawImages/wfc3/SBC')

# FLC Glob Pattern
FLC_CR_GLOB_PAT = '*_flt.fits'

# Define Max Sep
MAX_SEP = 0.8 * u.arcsec

## Load the Data

In [ ]:
# Get the File Names and Sort them by filter
fileNameDict = {}
for fn in DATA_DIR.rglob(FLC_CR_GLOB_PAT):

    # Open the file to get the filter
    with fits.open(fn) as hduList:
        hdr = hduList[0].header  # Get the Header
        if 'FILTER' in hdr:      # If the FILTER keyword exists (WFC3)
            filt = hdr['FILTER']
        elif 'CLEAR' not in hdr['FILTER1']:  # If FILTER1 is not clear (ACS)
            filt = hdr['FILTER1']
        else:                                # Else FILTER2 must be the filter (ACS)
            filt = hdr['FILTER2']

    # Store the Name using the filter as the dict key
    # Start the Empty List if Key does not exist
    if filt not in fileNameDict:
        fileNameDict[filt] = []
    fileNameDict[filt].append(fn)
    logger.debug("Added file %s to filter %s list.", fn.name, filt)
fileNameDict

## Align Images to GAIA

Sometimes, the FLCs have so many CRs and so few Milky Way stars, it makes aligning the images difficult.
In this case, a different strategy is to align the Hubble Pipeline DRCs/DRZs to GAIA using `TweakReg` then
assigning the found WCS solution to the associated FLCs/FLTs using `TweakBack`.

For an example of this strategy, consider the work documented in the [Abell 1367](https://github.com/wwaldron/a1367)
repo on GitHub where the [Image Reducer Notebook](https://github.com/wwaldron/a1367/blob/main/Images/ImageReducer.ipynb)
implements this methodology.

### Transfer WFC3/UVIS Parallel Alignment to SBC

ACS/SBC has no detectable stellar sources to align directly to GAIA. However, the WFC3/UVIS
parallel exposures were taken **simultaneously** with the SBC exposures, share the same HST
telescope pointing, and have already been aligned to the GAIA catalog using `TweakReg`
(with `updatehdr=True`). The GAIA-corrected WCS is stored as the primary WCS
(`WCSNAME: GAIA`) in each WFC3 FLC, and the original pipeline WCS is stored as the alternate
`A` WCS (`WCSNAME: IDC_*`).

The pointing correction is purely a telescope-level offset (ΔRA, ΔDec), so the same
angular shift applies to the SBC. The procedure is:

1. For each visit, compute the mean **ΔRA** and **ΔDec** from the WFC3 FLCs in that visit:
   `Δ = CRVAL (GAIA) − CRVAL_A (IDC)`.
2. Apply that mean offset to each SBC FLT file in the corresponding visit by updating
   `CRVAL1` and `CRVAL2` in the `SCI` extension and recording the new WCS name as `GAIA`.

Note: `TweakBack` is not applicable here — it backpropagates alignment from a drizzled product
to *its own input files* and cannot transfer corrections across detectors.


In [ ]:
# Shift Directory
SHIFT_DIR = Path('Shifts')

# Initialize the QTable
shifts = QTable(
    names=['filename', 'dx', 'dy', 'rot', 'scale', 'xrms', 'yrms'],
    dtype=[str, float, float, float, float, float, float]
)
shifts['dx'].unit = u.pix
shifts['dy'].unit = u.pix
shifts['xrms'].unit = u.pix
shifts['yrms'].unit = u.pix
shifts['rot'].unit = u.deg
units = [shifts[col].unit for col in shifts.colnames]

# Read the Shifts
for filename in SHIFT_DIR.glob('*.txt'):

    # Read Shift File
    shifts = vstack([shifts, QTable.read(filename, format='ascii.no_header', comment='#', names=shifts.colnames, units=units)])

# Add Visit Column
pattern = re.compile(r"visit(\d+)_")
shifts['visit'] = [int(re.search(pattern, fn).group(1)) for fn in shifts['filename']]

# Put Visit First
colnames = shifts.colnames
colnames.insert(0, colnames.pop(-1))
shifts = shifts[colnames]

# Sort
shifts.sort(['visit', 'filename'])
shifts

In [ ]:
# Get Shifts in Degrees
pixel_scale = []
for filename in shifts['filename']:
    with fits.open(filename) as hduList:
        wcs = WCS(hduList[1].header, hduList)
        pixel_scale.append(proj_plane_pixel_scales(wcs).mean() * u.deg/u.pix)

# Convert Pixel Scale to Quantity
pixel_scale = u.Quantity(pixel_scale)

# Change the Units of dx, dy, xrms, and yrms to Degrees
shifts['dx'] *= pixel_scale
shifts['dy'] *= pixel_scale
shifts['xrms'] *= pixel_scale
shifts['yrms'] *= pixel_scale
shifts

In [ ]:
# Get the Mean Shift per Visit in Degrees
mean_shifts = shifts.group_by('visit').groups.aggregate(np.mean)
mean_shifts

In [ ]:
# Apply Mean Shifts
for visit_dir in sorted(DATA_DIR.iterdir()):
    visit_match = re.search(r'visit_?(\d+)', visit_dir.name)
    if not visit_match:
        continue
    visit = visit_match.group(1)

    # Shift Row
    shift_row = mean_shifts[mean_shifts['visit'] == int(visit)]
    xsh, ysh = shift_row['dx'][0], shift_row['dy'][0]
    xrms, yrms = shift_row['xrms'][0], shift_row['yrms'][0]

    for fn in sorted(visit_dir.glob(FLC_CR_GLOB_PAT)):

        # Update WCS — archives old WCS as alternate 'A' before writing 'GAIA'
        updatehdr.updatewcs_with_shift(
            str(fn), str(fn),
            wcsname='GAIA',
            xsh=xsh.to_value(u.deg), ysh=ysh.to_value(u.deg),
            rot=0, scale=1,
            # xrms=xrms.to_value(u.deg), yrms=yrms.to_value(u.deg),
            reusename=True,
            force=True,
        )

        logger.info(
            "Updated %s (visit %s): dRA=%.5f\", dDec=%.5f\"",
            fn.name, visit, xsh.to_value(u.arcsec), ysh.to_value(u.arcsec),
        )


### TweakReg Data Quality Flags

This cell defines the [ACS DQ flags](https://www.stsci.edu/hst/instrumentation/acs/data-analysis/dq-flag-definitions) we
want to ignore in the TweakReg process. The DQ flags that are most often used are:

* [ACS DQ Flags](https://www.stsci.edu/hst/instrumentation/acs/data-analysis/dq-flag-definitions)
* [WFC3-UVIS DQ Flags](https://hst-docs.stsci.edu/wfc3dhb/chapter-3-wfc3-data-calibration/3-2-uvis-data-calibration-steps#id-3.2UVISDataCalibrationSteps-3.2.3DataQualityArrayInitialization)
* [WFC3-IR DQ Flags](https://hst-docs.stsci.edu/wfc3dhb/chapter-3-wfc3-data-calibration/3-3-ir-data-calibration-steps#id-3.3IRDataCalibrationSteps-3.3.1DataQualityInitialization)

For an example of how to implement multiple DQ flags, consider the
[Image Reducer for ESO 137-001](https://github.com/wwaldron/ESO-137-001/blob/main/Images/ImageReducer.ipynb).

<div class="alert alert-block alert-info">
    <b>Note:</b> In the cells below where <span style="font-family: 'Ariel', monospace;">TweakReg</span> is called,
    the user <i>must</i> update the <span style="font-family: 'Ariel', monospace;">updatehdr</span>
    keyword to <span style="font-family: 'Ariel', monospace;">True</span> and rerun the cell once a valid
    WCS solution is found.
    If the value is left as <span style="font-family: 'Ariel', monospace;">False</span>, the header in the
    input file will not be updated.
</div>

## Drizzle Images for CR Correction

Although there will be additional notes added later, it is worth noting that according to
[STScI](https://hst-docs.stsci.edu/drizzpac/chapter-6-reprocessing-with-the-drizzlepac-package/6-3-running-astrodrizzle#id-6.3RunningAstroDrizzle-SelectingtheOptimalScaleandPixfrac):

1. For sub-pixel dithered data, select an output scale that's smaller than the native scale.
It will even help in the cosmic ray rejection step.
1. A smaller final_pixfrac gives higher resolution and lower correlated noise, but also reduces
sensitivity to low-surface brightness features (though it is possible to convolve a high resolution
image later to go after low surface brightness features).
1. Keep the standard deviation of the weight map over the main part of the image to above ~0.3 of
the mean to insure that one does not lose significant signal-to-noise in ignoring the weight map in
final photometry.

To summarize the last step, a `final_scale`/`final_pixfrac` combo should be chosen such that,
for the weight image,
\begin{equation}
    \mathrm{std} \gtrsim 0.3 \, \mathrm{mean}
\end{equation}

### AstroDrizzle ACS Data Quality Flags

* [ACS DQ Flags](https://www.stsci.edu/hst/instrumentation/acs/data-analysis/dq-flag-definitions)
* [WFC3-UVIS DQ Flags](https://hst-docs.stsci.edu/wfc3dhb/chapter-3-wfc3-data-calibration/3-2-uvis-data-calibration-steps#id-3.2UVISDataCalibrationSteps-3.2.3DataQualityArrayInitialization)
* [WFC3-IR DQ Flags](https://hst-docs.stsci.edu/wfc3dhb/chapter-3-wfc3-data-calibration/3-3-ir-data-calibration-steps#id-3.3IRDataCalibrationSteps-3.3.1DataQualityInitialization)

In [ ]:
# DQ Bits
DQ_WARM_PIX = 64
DQ_BAD_COL  = 128
DQ_FULL_WELL= 256
DQ_SINK_PIX = 1024
DQ_GOOD_PIX = DQ_WARM_PIX + DQ_BAD_COL + DQ_FULL_WELL + DQ_SINK_PIX # Make these OK

In [ ]:
DQ_GOOD_PIX = 2**15 - 1  # All bits from 0 to 14 are good
# DQ_GOOD_PIX = 0  # No bits are good

### Drizzle F150LP Images

In [ ]:
# Drizzle Images
AstroDrizzle(
    [str(fn) for fn in fileNameDict['F150LP']],
    output='IC4040-F150LP-SBC',
    runfile='F150LP-Astro.log',
    wcskey='GAIA',
    context=False,
    configobj=None,
    num_cores=8,
    in_memory=False,
    build=True,
    restore=False,
    preserve=False,
    clean=True,
    skysub=False,
    driz_separate=False,
    median=False,
    blot=False,
    driz_cr=False,
    final_wht_type='EXP',
    final_pixfrac=1,
    final_bits=DQ_GOOD_PIX,
    final_wcs=True,
    final_rot=0,
    final_scale=0.03
)

### Drizzle F165LP Images

In [ ]:
# Drizzle Images
AstroDrizzle(
    [str(fn) for fn in fileNameDict['F165LP']],
    output='IC4040-F165LP-SBC',
    runfile='F165LP-Astro.log',
    wcskey='GAIA',
    context=False,
    configobj=None,
    num_cores=8,
    in_memory=False,
    build=True,
    restore=False,
    preserve=False,
    clean=True,
    skysub=False,
    driz_separate=False,
    median=False,
    blot=False,
    driz_cr=False,
    driz_cr_corr=False,
    final_wht_type='EXP',
    final_pixfrac=1,
    final_bits=DQ_GOOD_PIX,
    final_wcs=True,
    final_refimage='IC4040-F150LP-SBC_drz.fits'
)

In [ ]:
%%bash
# Move Log Files
mkdir -p logs/astrodrizzle
mv *.log logs/astrodrizzle

# Move Final Drizzled Images
mkdir -p ProcessedImages/HST/SBC/Drizzled
mv *_dr?.fits ProcessedImages/HST/SBC/Drizzled/